# Math LLM Probing & Evaluation Pipeline

This notebook implements a pipeline to:
1. Load `Qwen/Qwen2.5-Math-7B-Instruct`.
2. Generate answers to math problems (enforcing direct answers).
3. Extract hidden states.
4. Evaluate answers using Regex and SymPy.

In [1]:
import os
import sys

# 1. Clone the repo if it doesn't exist
REPO_NAME = "probing-llm-math"
if not os.path.exists(REPO_NAME):
    print("Cloning repository...")
    !git clone -b colab-pipeline-setup https://github.com/1hamzaiqbal/probing-llm-math.git
else:
    print(f"{REPO_NAME} already exists.")

# 2. Install dependencies
print("Installing dependencies...")
!pip install -q -r {REPO_NAME}/requirements.txt

# 3. Add to path so we can import src
if REPO_NAME not in sys.path:
    sys.path.append(REPO_NAME)

print("Setup complete!")

In [2]:
try:
    from src import model_utils, evaluator
    print("Successfully imported modules from src.")
except ImportError as e:
    print(f"Error importing modules: {e}")
    print("Current path:", sys.path)
    print("Current dir contents:", os.listdir("."))
    if os.path.exists(REPO_NAME):
        print(f"{REPO_NAME} contents:", os.listdir(REPO_NAME))

In [3]:
# [OPTIONAL] Reset & Update
# Run this cell if you want to pull the latest code and free up VRAM without restarting the runtime.

import gc
import torch
import importlib
import sys

# 1. Free VRAM
print("Clearing VRAM...")
try:
    del model
    del tokenizer
except NameError:
    pass

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("VRAM cleared.")

# 2. Pull latest code
print("Pulling latest code...")
!cd {REPO_NAME} && git pull

# 3. Reload modules
print("Reloading modules...")
import src.model_utils
import src.evaluator
importlib.reload(src.model_utils)
importlib.reload(src.evaluator)
from src import model_utils, evaluator

print("Modules reloaded. You can now re-run the 'Load Model' cell.")

In [4]:
# Load Model
# Use load_in_4bit=True for T4 GPUs to save VRAM and allow larger context
model, tokenizer = model_utils.load_model(model_name="Qwen/Qwen2.5-Math-7B-Instruct", load_in_4bit=True)

In [5]:
# # Define some test questions (Weird Formats & Hard Problems)
# questions = [
#     # Matrix Determinant (LaTeX array check)
#     {"question": "Compute the determinant of the matrix $\\begin{pmatrix} 1 & 2 \\\\ 3 & 4 \\end{pmatrix}$.", "answer": "-2"},
    
#     # Interval Notation (Set theory)
#     {"question": "Find the domain of $f(x) = \\sqrt{x-1}$.", "answer": "[1, \\infty)"},
    
#     # Coordinates (Tuple format)
#     {"question": "Find the intersection point of the lines $y=x$ and $y=2-x$.", "answer": "(1, 1)"},
    
#     # Hard Integral (Gaussian - Symbol check)
#     {"question": "Evaluate $\\int_{-\\infty}^{\\infty} e^{-x^2} dx$.", "answer": "\\sqrt{\\pi}"},
    
#     # Hard Number Theory (AIME 2001 #1 - Integer answer)
#     {"question": "Find the sum of all positive integers $n$ such that $n$ divides $n^2 + 1$.", "answer": "1"},
    
#     # Nested Surds (Weird LaTeX)
#     {"question": "Simplify \\sqrt{3 + 2\\sqrt{2}}.", "answer": "1+\\sqrt{2}"},
    
#     # Units / Physics Style
#     {"question": "If a car travels 60 miles in 2 hours, what is its average speed in mph?", "answer": "30"},
    
#     # Multiple Choice Style (Letter extraction)
#     {"question": "Which of the following is prime? (A) 4 (B) 9 (C) 11 (D) 15. Answer with the letter only.", "answer": "C"}
# ]

In [6]:
# # Run Pipeline
# results = []

# for item in questions:
#     q = item["question"]
#     gt = item["answer"]
    
#     print(f"Processing: {q}")
    
#     # Generate
#     pred_text, hidden_states, is_truncated = model_utils.generate_answer(model, tokenizer, q)
    
#     # Check truncation
#     if evaluator.check_truncation(pred_text, is_truncated):
#         print("  WARNING: Output truncated!")
    
#     # Extract clean answer
#     clean_pred = evaluator.extract_answer(pred_text)
    
#     # Evaluate
#     is_correct = evaluator.is_equivalent(clean_pred, gt)
    
#     # print(f"  Pred: {pred_text}") # Commented out to reduce clutter if needed
#     print(f"  Clean: {clean_pred}")
#     print(f"  Correct: {is_correct}")
    
#     results.append({
#         "question": q,
#         "ground_truth": gt,
#         "prediction": pred_text,
#         "clean_prediction": clean_pred,
#         "correct": is_correct,
#         "truncated": is_truncated
#     })

In [7]:
# # Analyze Results
# correct_count = sum(1 for r in results if r['correct'])
# accuracy = correct_count / len(results)
# print(f"Accuracy: {accuracy:.2%}")

# Linear Probing

This section runs the linear probing experiments.
1. Collect hidden states and correctness labels from GSM8K.
2. Train a logistic regression classifier to predict correctness.
3. Visualize the results.

In [8]:
# # 1. Collect Data
# # We'll use the MATH dataset (all levels) and enforce balancing
# # to ensure we get both correct and incorrect answers.
# # Removing 'Level 5' filter to allow easier questions so we can find correct answers too.

# !python3 probing-llm-math/src/collect_probe_data.py --num_samples 5 --dataset math --balance --audit_file probe_audit.csv

In [9]:
# # [Optional] View Audit Log
# import pandas as pd
# if os.path.exists("probe_audit.csv"):
#     df_audit = pd.read_csv("probe_audit.csv")
#     display(df_audit.head(20))
#     print("\nClass Distribution:")
#     print(df_audit['is_correct'].value_counts())
# else:
#     print("Audit file not found.")

In [10]:
# SECOND MINI RUN
!python probing-llm-math/src/collect_probe_data.py \
    --num_samples 150 \
    --dataset math \
    --balance \
    --output_file probe_data.pt

In [11]:
# # FIRST MINI RUN
# # 1. Collect probe data (with all metadata)
# !python probing-llm-math/src/collect_probe_data.py \
#     --num_samples 10 \
#     --dataset math \
#     --balance \
#     --output_file probe_data.pt

# # 2. Train all probes
# !python probing-llm-math/src/train_probe.py \
#     --data_file probe_data.pt \
#     --output_dir probe_results

# # 3. Generate heatmap (separate run, more samples)
# !python probing-llm-math/src/stratified_eval.py \
#     --samples_per_cell 5 \
#     --output_dir eval_results

In [12]:
# import os
# from IPython.display import Image, display

# print("--- Evaluation Results ---")
# result_dir = "eval_results"

# # Print summary.txt content
# summary_path = os.path.join(result_dir, "summary.txt")
# if os.path.exists(summary_path):
#     with open(summary_path, "r") as f:
#         summary_text = f.read()
#     print(summary_text)
# else:
#     print(f"(summary not found at {summary_path})")

# # Visualize heatmap
# heatmap_path = os.path.join(result_dir, "accuracy_heatmap.png")
# if os.path.exists(heatmap_path):
#     print("Displaying accuracy heatmap:")
#     display(Image(filename=heatmap_path))
# else:
#     print(f"(heatmap not found at {heatmap_path})")

# # Optionally confirm number of results
# results_pt_path = os.path.join(result_dir, "results.pt")
# if os.path.exists(results_pt_path):
#     print(f"\nSaved results file exists: {results_pt_path}")
# else:
#     print("(results.pt not found)")



In [13]:
# # 2. Train Probes & Analyze

# !python3 probing-llm-math/src/train_probe.py

In [14]:
# # 3. Visualize Results
# from IPython.display import Image, display
# import os

# results_dir = "probe_results"
# if os.path.exists(results_dir):
#     print("Probe Accuracy by Layer:")
#     display(Image(filename=f"{results_dir}/probe_accuracy.png"))
    
#     # Check for PCA plot (might vary by layer index)
#     pca_files = [f for f in os.listdir(results_dir) if f.startswith("pca_layer")]
#     if pca_files:
#         print("PCA Visualization:")
#         display(Image(filename=f"{results_dir}/{pca_files[0]}"))
# else:
#     print("No results found. Did the training script run successfully?")